<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Manipulation de données satellitaires - Productions Eumetsat - Format TIFF</div>

---

## 🎯 **Objectifs du TP**

##  🔎 Analyser une image satellitaire TIFF géoréférencée
##  🔄 Reprojeter et découper l'image
##  🎨 Ajouter des couches (contours) et des annotations (texte, logo)
##  🎬 Produire une animation à partir de plusieurs images

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise principalement la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **TIFF**, issus de la production opérationnelle disponible sur Eumetview.


### Qu'est-ce qu'ImageMagick ?
ImageMagick est une suite **logicielle libre et open source** dédiée à la **création**, la **conversion**, la **modification** et l'**analyse** d'**images raster**. C'est un outil très utilisé en automatisation, traitement d'images scientifiques, développement web et production graphique. </br>
Il permet de manipuler plus de 200 formats d'images, notamment : JPEG, PNG, TIFF, GIF, PDF, SVG... </br>
Il fonctionne principalement en ligne de commande, mais peut aussi être intégré dans des scripts ou des applications via des bibliothèques disponibles en C, C++, Python, PHP, Java, etc. </br>
https://imagemagick.org</br>

---

## 📂 **Fichier utilisé**

Le fichier TIFF étudié est un produit de type **Geocolor** du satellite Meteosat-12 :

📁 `/stockage/DATA/eumetview/20260614_geocolor_rgb_eumetview.tif`

> 💡 *Ce fichier est un produit téléchargé sur le site Eumetview.*

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

Le premier fichier TIF proposé est le suivant : 20260614_geocolor_rgb_eumetview.tif </br>
Il se trouve dans ce répertoire : **`/stockage/DATA/eumetview/`**</br></br>
❗ Attention : ce chemin doit-être modifié pour toute arborescence différente

In [ ]:
INPUT="/stockage/DATA/eumetview/20260614_geocolor_rgb_eumetview.tif"
print(f"📁 Fichier source configuré : {INPUT}")

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Visualisation du TIFF et de son contenu</h3>
</div>

La commande `gdalinfo` affiche les métadonnées géospatiales d’un fichier raster. Dans Jupyter, on la précède de `!`.

In [ ]:
!gdalinfo $INPUT

## 📋 Décryptage des informations clés (fichier mis à jour)

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Driver** | GTiff/GeoTIFF | Format standard géoréférencé (inchangé) |
| 📛 **Fichier** | `20260614_geocolor_rgb_eumetview.tif` | Produit **EUMETVIEW Geocolor**, date **2026-06-14 12:00 UTC** |
| 📐 **Taille** | **11408 × 10857** pixels | **Image rectangulaire** (non carrée, contrairement au disque complet) |
| 🛰️ **Projection** | **GEOGCRS WGS 84 (EPSG:4326)** | **Coordonnées géographiques (Lat/Lon)**. Projection équirectangulaire découpée. |
| 📍 **Étendue géographique** | Lon : -81.27° à 81.28°<br>Lat : -77.35° à 77.35° | Couvre une large bande du globe (découpe rectangulaire). |
| 🧭 **Origine (coin haut-gauche)** | **(-81.2778°, 77.3564°)** | Exprimée en **degrés décimaux** (et non en mètres). |
| 📏 **Pixel Size** | **(0.0142495°, -0.0142495°)** | Résolution d'**~1,6 km/pixel** à l'équateur (car 0.014° × 111 km ≈ 1,58 km). |
| 🏷️ **Métadonnées spécifiques** | `TIFFTAG_DATETIME=2026:06:14 12:00:00` |  |
| 🕒 **Date image** | 2026-06-14 12:00:00 UTC | Heure du slot |
| 🗜️ **Compression / Structure** | **DEFLATE** (sans perte)<br>Interleave = PIXEL | **Changement majeur** : DIfférent par rapport à Météo-France du JPEG/YCbCr. C'est du RGB natif compressé sans perte (qualité maximale). |
| 🎨 **Bandes** | Rouge, Vert, Bleu | 8 bits par canal, **NoData = 0** (zones hors image en noir pur). |
| 🗂️ **Pyramides (Overviews)** | **5704×5429, 2852×2715, 1426×1358, 713×679** | Versions réduites pour l'affichage rapide (adaptées au nouveau format rectangulaire). | 


💡Remarque : la résolution est ici environ affichée à 1,6km en raison de la projection EPSG:4326

In [ ]:
cd ~/MF_DATA_MANIPULATION

📁 **Création du dossier de résultats**

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'

## 🖼️ **Redimensionnement et conversion JPG**

Nous allons redimensionner l'image et la convertir en JPG. L'utilisation de l'overview `[3]` (image 700x700 pixels) permet un gain de temps.

In [ ]:
!convert -quiet -resize 700x700 $INPUT[3]  $output/20260614_geocolor_rgb_eumetview_700x700.jpg

### 🖼️ **Visualisation de l'image**

In [ ]:
im = Image.open(output + '/20260614_geocolor_rgb_eumetview_700x700.jpg', 'r')
display(im)

📌 **Description** : Image **Geocolor RGB** en vue 4326, depuis le satellite **Meteosat-12** positionné à 0° de longitude. Ce produit est réalisé à partir de fichiers chunks, avec diverses corrections (Diffusion, parallaxe, correction angle solaire).

> 💡 *Le produit Geocolor RGB combine plusieurs canaux pour restituer une image en "vraies couleurs" jour et nuit.*

<div class="alert alert-info alert-success">
<h3> 2 - 🌐 Comparaison des formats d'image</h3>
</div>

| Format | Compression | Poids | Géoréférencement | Utilisation |
|--------|-------------|-------|------------------|-------------|
| **JPG** | Avec pertes (défaut 92) | Léger | ❌ Non | Web, visualisation rapide |
| **PNG** | Sans pertes | Lourd (4× JPG) | ❌ Non | Archives, transparence |
| **TIF** | Optionnelle (LZW, JPEG) | Très lourd (7× JPG) | ✅ Oui | SIG, traitement géospatial |

> ⚠️ *Seul le format TIF conserve le géoréférencement (coordonnées, projection).*</br>
> ⚠️ *Même avec une compression à 100, le format JPEG crée une altération.*</br>
> *Les différences visuelles sont souvent très ténues, voire invisibles, l'interpolation JPG par défaut "Lanczos" est un très bon compromis netteté / aliasing (pixellisation visible).*

In [ ]:
!convert -resize 700x700 $INPUT[3]  $output/20260614_geocolor_rgb_eumetview_700x700.jpg  >/dev/null 2>&1 
!convert -resize 700x700 $INPUT[3]  $output/20260614_geocolor_rgb_eumetview_700x700.tif  >/dev/null 2>&1 
!convert -resize 700x700 $INPUT[3]  $output/20260614_geocolor_rgb_eumetview_700x700.png  >/dev/null 2>&1 
!identify -quiet -format "%f : %b\n" $output/20260614_geocolor_rgb_eumetview_700x700.jpg $output/20260614_geocolor_rgb_eumetview_700x700.tif $output/20260614_geocolor_rgb_eumetview_700x700.png

📊 **Observations** :
- Le format **PNG** est environ **4 fois plus lourd** que le JPG
- Le format **TIF** est environ **7 fois plus lourd** que le JPG

❗ **Attention** : La commande `convert` (ImageMagick) **ne conserve pas le géoréférencement**.

In [ ]:
!gdalinfo $output/20260614_geocolor_rgb_eumetview_700x700.jpg 2>/dev/null | head -20

### 🔧 **Pour conserver le géoréférencement : `gdalwarp`**

La commande **`gdalwarp`** redimensionne l'image tout en préservant les informations géospatiales.

In [ ]:
!gdalwarp -overwrite -ts 700 700 $INPUT  $output/20260614_geocolor_rgb_eumetview_700x700.tif  >/dev/null 2>&1

In [ ]:
!gdalinfo $output/20260614_geocolor_rgb_eumetview_700x700.tif 2>/dev/null | head -25

✅ **Succès** : Les métadonnées de géoréférencement sont bien conservées.

### 🗜️ **Compression des fichiers TIF**

| Compression | Type | Gain | Usage |
|-------------|------|------|-------|
| **LZW** | Sans perte | ~25% | Archives, données précises |
| **JPEG/YCbCr** | Avec pertes (qualité 95) | ~80% | Diffusion, web, visualisation |
| **DEFLATE** | **Sans perte** | **Jusqu'à ~55-65%** | **Archivage, données scientifiques, traitements intermédiaires** |

> 💡 *DEFLATE est l'algorithme utilisé par les fichiers ZIP et PNG. Il est sans perte : l'image décompressée est **identique pixel par pixel** à l'originale.*
>
> Son efficacité dépend des données : avec un **prédicteur** (qui exploite les corrélations entre pixels), le gain peut atteindre **55%**. C'est un bon compromis entre taux de compression et vitesse, recommandé pour les images satellitaires.* 

> 💡 *La compression YCbCr JPEG95 (Luminance/couleur) est très efficace. Le gain en poids est important, mais il y a des pertes (infimes, souvent invisibles, mais existantes)*</br>

In [ ]:
!gdalwarp -overwrite -ts 700 700 $INPUT $output/20260614_geocolor_rgb_eumetview_700x700_compress_YCbCr.tif -co COMPRESS=JPEG -co JPEG_QUALITY=95 -co PHOTOMETRIC=YCBCR >/dev/null 2>&1
!gdalwarp -overwrite -ts 700 700 $INPUT $output/20260614_geocolor_rgb_eumetview_700x700_compress_LZW.tif -co COMPRESS=LZW -co PREDICTOR=2  >/dev/null 2>&1
!gdalwarp -overwrite -ts 700 700 $INPUT $output/20260614_geocolor_rgb_eumetview_700x700_compress_deflate.tif -co COMPRESS=DEFLATE -co ZLEVEL=9 -co PREDICTOR=2  >/dev/null 2>&1
!identify -quiet -format "%f : %b\n" $output/20260614_geocolor_rgb_eumetview_700x700.tif
!identify -quiet -format "%f : %b\n" $output/20260614_geocolor_rgb_eumetview_700x700_compress_YCbCr.tif
!identify -quiet -format "%f : %b\n" $output/20260614_geocolor_rgb_eumetview_700x700_compress_LZW.tif
!identify -quiet -format "%f : %b\n" $output/20260614_geocolor_rgb_eumetview_700x700_compress_deflate.tif

📊 **Résultats** :
- **LZW** : Poids ≈ 75% du fichier non compressé
- **YCbCr (JPEG)** : Poids ≈ 20% du fichier non compressé

> 👉 La compression est un paramètre très important à prendre en compte, que cela soit en ce qui concerne la perte d'information, ou le volume à manipuler.

> 💡 *Une image Meteosat-12 plein disque à 500 m en TIF compressé YCbCr pèse environ **160 Mo**, contre **1,4 Go** non compressée.*

Dans GDAL / GeoTIFF, -co PREDICTOR=2 est une option d’amélioration de compression utilisée avec certains algorithmes comme LZW ou DEFLATE. </br>
(Comme dans beaucoup d’images (MNT, satellite, photos…), les pixels voisins se ressemblent, les différences sont souvent petites et répétitives. Le prédicteur encode les différences entre pixels voisins)</br>
LZW + PREDICTOR=2 : La taille sera environ de 70–90% par rapport à la compression sans PREDICTOR.

<div class="alert alert-info alert-success">
<h3> 3 - 🌐 Reprojections</h3>
</div>

`gdalwarp` permet de changer la projection d'une image.

🛰️ Projection GEOS (Geostationary Earth Orbit Satellite)

Système de coordonnées projetées en mètres, représentant la vue d'un satellite géostationnaire. Visuellement : azimutale (disque terrestre vu de l'espace).

· Principe : Simule la façon dont un satellite géostationnaire balaie la Terre à intervalles angulaires réguliers.
· Satellites concernés : Météosat (Europe), GOES (États-Unis), Himawari (Japon), etc.
· Paramètres essentiels :
  · +h : Hauteur du satellite au-dessus de l'ellipsoïde (en mètres). Pour Météosat : 35785831 m.
  · +lon_0 : Longitude du point subsatellite (en degrés).
  · +a & +b : Demi-grand axe et demi-petit axe de l'ellipsoïde (en mètres).
· Axes de balayage (+sweep) : Définit l'axe de rotation. y (défaut) pour Météosat, x pour GOES.

💡 Les coordonnées projetées (en mètres) se convertissent en angle de balayage (en radians) par la relation : angle = coordonnée / h. La taille des pixels est donc homogène en mètres sur l'ensemble du disque, contrairement à une projection géographique. 

In [ ]:
!gdalwarp -overwrite -r cubic -t_srs '+proj=geos +a=6378169 +b=6356583.8 +lon_0=0 +h=35785834 +x_0=0 +y_0=0' -te -5500000 -5500000 5500000 5500000 -ts 700 700  $output/20260614_geocolor_rgb_eumetview_700x700_compress_LZW.tif $output/20260614_geocolor_rgb_eumetview_700x700_geos.tif  >/dev/null 2>&1
!convert  $output/20260614_geocolor_rgb_eumetview_700x700_geos.tif $output/20260614_geocolor_rgb_eumetview_700x700_geos.jpg
imGeos = Image.open(output + '/20260614_geocolor_rgb_eumetview_700x700_geos.jpg', 'r')
display(imGeos)

### 🗺️ **Projection EPSG:4326 (WGS84 - longitude/latitude)**

Système de coordonnées géographiques en degrés. Visuellement : **cylindrique équidistante** (planisphère "plate").</br>
*WGS : World Geodetic System 1984,*</br>
*EPSG:4326 : Système géographique*

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 $output/20260614_geocolor_rgb_eumetview_700x700_geos.tif $output/20260614_geocolor_rgb_eumetview_700x700_4326.tif >/dev/null 2>&1
!gdalwarp -overwrite -t_srs EPSG:4326 $output/20260614_geocolor_rgb_eumetview_700x700_4326.tif $output/20260614_geocolor_rgb_eumetview_700x700_4326.jpg >/dev/null 2>&1
!gdalinfo $output/20260614_geocolor_rgb_eumetview_700x700_4326.tif |grep GEOGCRS

On retrouve le système  WGS 84, les coordonnées en latitude / longitude (degrés)

In [ ]:
im2 = Image.open(output + '/20260614_geocolor_rgb_eumetview_700x700_4326.jpg', 'r')
display(im2)

---

### 🧭 **Projection Mercator (EPSG:3395)**

Projection cylindrique **conforme** (conservation des angles). </br>
Référence dans le monde grâce à sa précision pour les voyages marins du fait de la conservation des angles qu'elle permet, essentielle pour prévoir les routes de navigation.

| EPSG | Nom | Utilisation |
|------|-----|-------------|
| 3395 | Mercator (ellipsoïde) | Données précises |
| 3857 | Web Mercator (sphérique) | Google Maps, OpenStreetMap |

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:3395 $output/20260614_geocolor_rgb_eumetview_700x700.tif $output/20260614_geocolor_rgb_eumetview_700x700_3395.jpg >/dev/null 2>&1
im3 = Image.open(output + '/20260614_geocolor_rgb_eumetview_700x700_3395.jpg', 'r')
display(im3)

> 💡 *On peut aussi utiliser `-t_srs '+proj=merc'` comme alternative à `EPSG:3395` qui donnera le même résultat.*</br>

#gdalwarp -overwrite -t_srs '+proj=merc' ...

---

### 🌍 **Projection de Peters (Gall-Peters)**

Projection cylindrique équivalente qui respecte les **superficies réelles** des continents (contrairement à Mercator).

In [ ]:
!gdalwarp -overwrite -t_srs "+proj=cea +lon_0=0 +lat_ts=45 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs" $output/20260614_geocolor_rgb_eumetview_700x700.tif $output/20260614_geocolor_rgb_eumetview_700x700_peter.jpg >/dev/null 2>&1
im3 = Image.open(output + '/20260614_geocolor_rgb_eumetview_700x700_peter.jpg', 'r')
display(im3)

<div class="alert alert-info alert-success">
<h3> 4 - ✂️ Découpes (cropping)</h3>
</div>

### 🇫🇷 **Découpe centrée sur Casablanca (projection EPSG:4326)**

L'avantage de la projection WGS84 est de pouvoir spécifier les limites en degrés.

**Options importantes** :
- `-t_srs {proj}` : nouvelle projection
- `-te {west} {south} {east} {north}` : étendue de découpe (W S E N)
- `-ts {width} {height}` : taille de sortie en pixels
- `-tr {xres} {yres}` : résolution spatiale

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 -te -10 33 -3 37  $INPUT $output/20260614_geocolor_rgb_eumetview_casablanca.tif #>/dev/null 2>&1
!convert -resize 900 $output/20260614_geocolor_rgb_eumetview_casablanca.tif $output/20260614_geocolor_rgb_eumetview_900_casablanca.jpg #>/dev/null 2>&1
im4 = Image.open(output + '/20260614_geocolor_rgb_eumetview_900_casablanca.jpg', 'r')
display(im4)

### 🛰️ **Projection orthographique (vue de l'espace)**

La projection orthographique donne l'impression de regarder la Terre depuis l'espace, à la verticale d'un point.

**Paramètres** :
- `lat_0` : latitude du centre
- `lon_0` : longitude du centre
- `-te` : étendue en mètres (optionnel)

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=35 lon_0=-7.5' -ts 960 540 -te -480000 -270000 480000 270000 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_casablanca.jpg >/dev/null 2>&1
im6 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca.jpg', 'r')
display(im6)


> ⚠️ **Attention** : L'image peut être déformée si le ratio de sortie (`-ts`) ne correspond pas au ratio géographique de la zone découpée.</br>
> Ici, la taille 960x540 correspond a du 16/9.

<div class="alert alert-info alert-success">
<h3> 5 - ✂️ Résolution, déformation & rééchantillonnage </h3>
</div>

📏 **Résolution** : Dans cet exemple, la résolution est d'environ 2333 m/pixel (perte par rapport à l'original).
-> Nous avons donc perdu en résolution spatiale

### 🔍 **Avec et sans imposer la taille (`-ts`)**

-> Pour garder la pleine résolution : sans option de taille de sortie -ts

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=35 lon_0=-7.5' -te -480000 -270000 480000 270000 -ts 192 108 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_min.jpg >/dev/null 2>&1
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=35 lon_0=-7.5' -te -480000 -270000 480000 270000 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_HD.jpg >/dev/null 2>&1
!identify $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_min.jpg
!identify $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_HD.jpg

In [ ]:
immin = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca_min.jpg', 'r')
display(immin)
imhd = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca_HD.jpg', 'r')
display(imhd)

❌ Attention à ne pas déformer l'image avec un mauvais ratio!

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=35 lon_0=-7.5' -te -480000 -270000 480000 270000 -ts 800 200 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_min2.jpg >/dev/null 2>&1
immin2 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca_min2.jpg', 'r')
display(immin2)

### 🎚️ **Imposer une résolution spatiale avec `-tr`**

Cette option permet de modifier la résolution d'une image, en sous/sur-échantillonnant</br>
Exemple : -tr 1000 1000 pour un pixel de 1 000 mètres

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=35 lon_0=-7.5' -tr 1000 1000 -te -480000 -270000 480000 270000 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_tr.jpg >/dev/null 2>&1
!identify $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_HD.jpg
!identify $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_tr.jpg
im6 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca_tr.jpg', 'r')
display(im6)

### 🔎 L'image produite sur la zone de Casablanca avec l'option -tr 1000 1000 est plus grande que l'image pleine résolution

### 🔍 **Comparaison des images : zoom sur le détroit de Gibraltar**

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=36 lon_0=-5.6' -te -160000 -90000 160000 90000 $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom.jpg >/dev/null 2>&1
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=36 lon_0=-5.6' -tr 3000 3000 -te -160000 -90000 160000 90000 $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_tr_zoom.jpg >/dev/null 2>&1
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=36 lon_0=-5.6' -ts 80 45 -te -160000 -90000 160000 90000 $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_ts_zoom.jpg >/dev/null 2>&1
im7 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom.jpg', 'r')
print("🏭 option -te")
display(im7)
im8 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_tr_zoom.jpg', 'r')
print("🏭 option -tr")
display(im8)
im9 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_ts_zoom.jpg', 'r')
print("🏭 option -ts")
display(im9)

* Agrandissement des images pour comparer:

In [ ]:
!convert -quiet -resize 700 $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom.jpg  $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom.jpg >/dev/null 2>&1
!convert -quiet -resize 700 $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_tr_zoom.jpg  $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_tr_zoom.jpg >/dev/null 2>&1
!convert -quiet -resize 700 $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_ts_zoom.jpg $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_ts_zoom.jpg >/dev/null 2>&1
im10 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom.jpg', 'r')
print("🏭 option -te")
display(im10)
im11 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_tr_zoom.jpg', 'r')
print("🏭 option -tr")
display(im11)
im12 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_ts_zoom.jpg', 'r')
print("🏭 option -ts")
display(im12)

### 🎨 **Méthodes de rééchantillonnage : avec et sans (`-r`)**

| Option | Description | Usage |
|--------|-------------|-------|
| `near` | Plus proche voisin | Rapide, préserve les valeurs (données catégorielles) |
| `bilinear` | Bilinéaire | Lissage modéré, bon compromis |
| `cubic` | Cubique (défaut recommandé) | Lisse et net, idéal pour images |
| `lanczos` | Lanczos | Très net mais plus lent, peut créer des artefacts |

> 💡 **Recommandation** : Utilisez `-r cubic` pour un bon compromis entre netteté et lissage.

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=36 lon_0=-5.6'  -ts 800 450 -te -80000 -45000 80000 45000 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom_-rcubic.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=36 lon_0=-5.6'  -ts 800 450 -te -80000 -45000 80000 45000 $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom_.jpg
im13 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom_-rcubic.jpg', 'r')
display(im13)
im14 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_gibraltar_HD_zoom_.jpg', 'r')
display(im14)

> 💡 Pour **comparer**, télécharger les images et utiliser le **slider de la Galerie CMS** : http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html

📚 **Pour aller plus loin** :
- 📌 [Fiche produit Geocolor RGB](../DOCS/MTG_fiche_geocolor_RGB.pdf)
- 📌 [Recettes des produits RGB](../DOCS/mtg_fci_synthese_produits.pdf)
- 🛠️ [Visualisation interactive des bandes](http://e-sat.cms.meteo.fr/GALERIE/loupes.html)

<div class="alert alert-info alert-success">
<h3> 6 - 🖌️ Ajouter un contour, un logo, annoter l'image</h3>
</div>

### 🗺️ **Ajout de contours avec `gdal_rasterize`**

La commande `gdal_rasterize` permet de dessiner des contours (fichiers shapefile) sur une image.

**Options de couleur** :
- `-b 1 -burn 255 -b 2 -burn 255 -b 3 -burn 255` → blanc
- `-b 1 -burn 0 -b 2 -burn 0 -b 3 -burn 0` → noir
- `-b 1 -burn 255 -b 2 -burn 0 -b 3 -burn 0` → rouge

> ⚠️ *Attention : `gdal_rasterize` modifie directement le fichier d'entrée (le TIF est écrasé).*

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=35 lon_0=-7.5' -te -480000 -270000 480000 270000 -ts 960 540 -r cubic $INPUT $output/20260614_geocolor_rgb_eumetview_ortho_casablanca.tif
!gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp  $output/20260614_geocolor_rgb_eumetview_ortho_casablanca.tif >/dev/null 2>&1
!convert -quiet -resize 1920x1080 $output/20260614_geocolor_rgb_eumetview_ortho_casablanca.tif $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour.jpg
im18 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour.jpg', 'r')
display(im18)

### 📝 **Ajout d'un titre et d'un bandeau**

In [ ]:
titre="Meteosat-12 - Geocolor RGB - 14/06/2026 à 12 h UTC"
# Ajout d'un bandeau bleu semi-transparent
!composite -quiet -geometry +0+0 OUTILS/bandeau_bleu_1920x1080.png $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour.jpg $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour_bandeau.jpg
# Ajout du texte
!convert -resize 1920x1080 $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour_bandeau.jpg -font OUTILS/Police_Marianne/Marianne-Bold.otf -pointsize 25 -fill '#eeeeee' -annotate +24+24 "$titre" $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour_date.jpg >/dev/null 2>&1
im19 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour_date.jpg', 'r')
display(im19)

### 🏷️ **Ajout du logo Eumetsat**

In [ ]:
!composite -quiet -geometry +1520+931 OUTILS/EUMETSAT_logo.png $output/20260614_geocolor_rgb_eumetview_ortho_casablanca_contour_date.jpg $output/20260614_geocolor_rgb_eumetview_ortho_casablanca.jpg
im20 = Image.open(output + '/20260614_geocolor_rgb_eumetview_ortho_casablanca.jpg', 'r')
display(im20)

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

Toujours à partir de :
  - ce même fichier tiff, ou </br>
  - du fichier tiff Geocolor RGB 20260622_geocolor_rgb_eumetview/tif : </br>
    
-> Découper ce tiff sur la zone du sud-est du Niger  : choisir le domaine et la projection, convertir en jpg redimensionné </br>
-> Bonus : ajouter les frontières et un titre sur l'image


## 🌟 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Documentation GDAL](https://gdal.org/en/stable/) | Référence complète de GDAL |

---

✅ **Fin du TP**